# RNA Velocity — HSC compartment

scVelo (dynamical model) + CellRank fate probabilities for the HSC compartment.

**Prerequisites**
1. Run `./scripts/run_velocyto.sh` to generate per-sample loom files
   (`data/mpn_10x_camelia/velocyto/<sample>.loom`).
2. Run `targets::tar_make()` so `hsc_h5ad_path` is built
   (`output/objects/hsc.h5ad`).

**Workflow**
1. Load Seurat-exported h5ad (UMAP, PCA, clusters, metadata)
2. Load & merge velocyto loom files (spliced / unspliced / ambiguous)
3. scVelo dynamical model → velocity graph → latent time
4. CellRank: velocity + connectivity kernels → GPCCA → fate probabilities

## 1  Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import anndata as ad
import scanpy as sc
import scvelo as scv
import cellrank as cr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

scv.settings.verbosity = 3
scv.settings.presenter_view = True
sc.settings.set_figure_params('scvelo', dpi=100, dpi_save=200, transparent=True)

print('scvelo version :', scv.__version__)
print('cellrank version:', cr.__version__)

## 2  Configuration

In [ ]:
# ── Paths (relative to repo root; adjust if running outside container) ────────
REPO_ROOT  = Path('../')                          # notebooks/ is one level below root
H5AD_PATH  = REPO_ROOT / 'output/objects/hsc.h5ad' # exported by hsc_h5ad_path target
LOOM_DIR   = REPO_ROOT / 'data/mpn_10x_camelia/velocyto'
OUT_DIR    = REPO_ROOT / 'output/objects/velocity'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Parameters ────────────────────────────────────────────────────────────────
N_TOP_GENES   = 2000   # variable genes for scVelo
MIN_SHARED    = 30     # min spliced+unspliced counts per gene
N_PCS         = 30     # PCs for moments
N_NEIGHBORS   = 30     # neighbours for moments
N_JOBS        = 8      # parallel jobs for dynamics / velocity graph
N_MACROSTATES = 6      # initial GPCCA macrostates (tune after inspection)

# ── Cluster / metadata columns (from Seurat export) ──────────────────────────
CLUSTER_COL   = 'seurat_clusters'   # update to the actual cluster column name
SAMPLE_COL    = 'sample.name'
CONDITION_COL = 'condition'         # WT / Het_mChNeg / Het_mChPos

## 3  Load data

### 3.1  Seurat-exported h5ad
Carries UMAP (`X_umap`), PCA (`X_pca`), cluster labels, and SCT-normalised counts.

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print('\nobs columns:', list(adata.obs.columns))
print('obsm keys:  ', list(adata.obsm.keys()))

In [ ]:
# Rename cluster column if needed — inspect output above and adjust CLUSTER_COL
# Seurat's cluster column is stored in obs after SCE conversion;
# it may be named e.g. 'sNN_pca.SCT_res.0.1' — rename for convenience.
cluster_cols = [c for c in adata.obs.columns if 'res.0.1' in c or c == 'seurat_clusters']
if cluster_cols and cluster_cols[0] != CLUSTER_COL:
    adata.obs.rename(columns={cluster_cols[0]: CLUSTER_COL}, inplace=True)
    print(f'Renamed {cluster_cols[0]!r} → {CLUSTER_COL!r}')

adata.obs[CLUSTER_COL] = adata.obs[CLUSTER_COL].astype(str).astype('category')

sc.pl.umap(adata, color=CLUSTER_COL, legend_loc='on data', title='HSC clusters')

### 3.2  Velocyto loom files (spliced / unspliced / ambiguous)

In [ ]:
loom_files = sorted(LOOM_DIR.glob('*.loom'))
print(f'Found {len(loom_files)} loom files:')
for f in loom_files:
    print(' ', f.name)

In [ ]:
# Read and concatenate loom files
looms = [scv.read_loom(str(f), validate=False) for f in loom_files]
loom  = ad.concat(looms, merge='same')
print('Loom combined:', loom)

### 3.3  Merge: add spliced / unspliced to adata

Velocyto barcodes follow `<sample>:<barcode>x`; Seurat barcodes are `<sample>_<barcode>`.  
`scv.utils.clean_obs_names` strips sample prefixes/suffixes — match on the bare barcode.

In [ ]:
scv.utils.clean_obs_names(adata)
scv.utils.clean_obs_names(loom)

adata = scv.utils.merge(adata, loom)
print('After merge:', adata)
print('Spliced layer shape:', adata.layers['spliced'].shape)

In [ ]:
# QC: fraction of cells that retained spliced counts after merge
n_matched = (adata.layers['spliced'].sum(1) > 0).sum()
print(f'{n_matched}/{adata.n_obs} cells have spliced counts ({100*n_matched/adata.n_obs:.1f}%)')

## 4  scVelo preprocessing

In [ ]:
scv.pp.filter_and_normalize(
    adata,
    min_shared_counts = MIN_SHARED,
    n_top_genes       = N_TOP_GENES,
    log               = True
)
scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=N_NEIGHBORS)

## 5  scVelo — dynamical model

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=N_JOBS)
scv.tl.velocity(adata, mode='dynamical')
scv.tl.velocity_graph(adata, n_jobs=N_JOBS)

In [ ]:
# Velocity stream on UMAP
scv.pl.velocity_embedding_stream(
    adata, basis='umap', color=CLUSTER_COL,
    title='HSC — velocity stream',
    save=str(OUT_DIR / 'velocity_stream.png')
)

In [ ]:
# Arrow embedding
scv.pl.velocity_embedding(
    adata, basis='umap', arrow_length=3, arrow_size=2,
    color=CLUSTER_COL, alpha=0.6,
    save=str(OUT_DIR / 'velocity_arrows.png')
)

### 5.1  Velocity confidence & coherence

In [ ]:
scv.tl.velocity_confidence(adata)
keys = ['velocity_length', 'velocity_confidence']
scv.pl.scatter(
    adata, c=keys, cmap='coolwarm',
    perc=[5, 95], size=60,
    save=str(OUT_DIR / 'velocity_confidence.png')
)

### 5.2  Latent time

In [ ]:
scv.tl.latent_time(adata)
scv.pl.scatter(
    adata, color='latent_time',
    color_map='gnuplot', size=80, perc=[2, 98],
    title='HSC — latent time',
    save=str(OUT_DIR / 'latent_time.png')
)

In [ ]:
# Top likelihood genes
top_genes = adata.var['fit_likelihood'].nlargest(10).index.tolist()
print('Top dynamic genes:', top_genes)
scv.pl.scatter(
    adata, basis=top_genes[:6], ncols=3,
    frameon=False, color='latent_time', color_map='gnuplot',
    save=str(OUT_DIR / 'top_dynamic_genes.png')
)

### 5.3  PAGA trajectory graph

In [ ]:
adata.uns['neighbors']['distances'] = adata.obsp['distances']
adata.uns['neighbors']['connectivities'] = adata.obsp['connectivities']

scv.tl.paga(adata, groups=CLUSTER_COL)
df_paga = scv.get_df(adata, 'paga/transitions_confidence', precision=2).T
print(df_paga.to_string())

scv.pl.paga(
    adata, basis='umap', size=50, alpha=0.1,
    min_edge_width=2, node_size_scale=1.5,
    save=str(OUT_DIR / 'paga.png')
)

## 6  CellRank — fate probabilities

Combine velocity and connectivity kernels, then use GPCCA to identify macrostates and compute fate probabilities.

In [ ]:
from cellrank.kernels import VelocityKernel, ConnectivityKernel

vk = VelocityKernel(adata)
vk.compute_transition_matrix()

ck = ConnectivityKernel(adata)
ck.compute_transition_matrix()

# 80% velocity / 20% connectivity
combined = 0.8 * vk + 0.2 * ck
print(combined)

### 6.1  Macrostate identification (GPCCA)

In [ ]:
from cellrank.estimators import GPCCA

g = GPCCA(combined)
g.fit(n_states=N_MACROSTATES, cluster_key=CLUSTER_COL)
g.plot_macrostates(which='all', basis='umap', legend_loc='right margin',
                   save=str(OUT_DIR / 'macrostates.png'))

In [ ]:
# Coarse-grained transition matrix between macrostates
g.plot_coarse_T(annotate=True, show_stationary_dist=True,
                save=str(OUT_DIR / 'coarse_T.png'))

In [ ]:
# Assign terminal and initial states — inspect macrostate plot above first
# then set terminal states manually.
# Example (update based on your data):
# g.set_terminal_states(['Macrostate_1', 'Macrostate_3'])
# g.set_initial_states(['Macrostate_0'])
g.predict_terminal_states()
g.plot_macrostates(which='terminal', basis='umap',
                   save=str(OUT_DIR / 'terminal_states.png'))

### 6.2  Fate probabilities

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(
    basis='umap', same_plot=False, ncols=3,
    save=str(OUT_DIR / 'fate_probabilities.png')
)

In [ ]:
# Fate probability by condition
if CONDITION_COL in adata.obs.columns:
    fate_cols = [c for c in adata.obs.columns if c.startswith('lineage_')]
    if fate_cols:
        fig, axes = plt.subplots(1, len(fate_cols),
                                 figsize=(5 * len(fate_cols), 4))
        if len(fate_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, fate_cols):
            adata.obs.boxplot(
                column=col, by=CONDITION_COL, ax=ax,
                grid=False, notch=False
            )
            ax.set_title(col)
            ax.set_xlabel('')
        plt.suptitle('Fate probability by condition')
        plt.tight_layout()
        fig.savefig(OUT_DIR / 'fate_by_condition.png', dpi=150)
        plt.show()

### 6.3  Driver gene identification

In [ ]:
# Genes correlated with fate probability (one set per terminal state)
g.compute_lineage_drivers(cluster_key=CLUSTER_COL, return_drivers=True)
g.plot_lineage_drivers(
    n_genes=8,
    save=str(OUT_DIR / 'lineage_drivers.png')
)

### 6.4  Gene expression trends along pseudotime

In [ ]:
# Canonical HSC / lineage markers — adjust to confirmed markers from R analysis
GENES_OF_INTEREST = [
    'Gata1', 'Pf4', 'Vwf',        # MK / erythroid
    'Csf1r', 'Ly6c2',              # monocyte
    'Cd34', 'Kit', 'Slamf1',       # HSC
    'Mpl',                         # MPN driver
]
# Keep only genes present in adata
genes_ok = [g for g in GENES_OF_INTEREST if g in adata.var_names]
print('Available genes:', genes_ok)

if genes_ok:
    model = cr.models.GAM(adata)
    cr.pl.gene_trends(
        adata, model=model,
        genes=genes_ok,
        data_key='X',
        ncols=3,
        save=str(OUT_DIR / 'gene_trends.png')
    )

## 7  Save

In [ ]:
out_h5ad = OUT_DIR / 'hsc_velocity.h5ad'
adata.write_h5ad(out_h5ad)
print(f'Saved: {out_h5ad}')